# 03 Модели

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm
import optuna
import json
from joblib import Parallel, delayed

In [2]:
RANDOM_STATE = 42
ID_COL     = 'customer_id'

## 1. Загрузка данных

In [3]:
df_target = pd.read_parquet('train_target.parquet')
target_cols = [c for c in df_target.columns if c != ID_COL]
print(f'Targets: {len(target_cols)}')

tree_nan_train = pd.read_parquet('tree_nan_train.parquet')
tree_zero_train = pd.read_parquet('tree_zero_train.parquet')
linear_train   = pd.read_parquet('linear_train.parquet')

print(f'tree_nan  : {tree_nan_train.shape}')
print(f'tree_zero : {tree_zero_train.shape}')
print(f'linear    : {linear_train.shape}')

Targets: 41
tree_nan  : (750000, 208)
tree_zero : (750000, 340)
linear    : (750000, 706)


# Проверим, что датасеты верно собрали

In [4]:
SAMPLE_N   = 50_000   # строк для обучения+валидации

In [5]:
# Выбираем один таргет для проверки
# Возьмём тот, у которого достаточно позитивных примеров (не слишком редкий)
pos_rates = df_target[target_cols].mean().sort_values(ascending=False)
TARGET = pos_rates[pos_rates > 0.01].index[0]  # первый таргет с >1% positive rate
print(f'Выбран таргет: {TARGET}  (positive rate = {pos_rates[TARGET]:.3%})')

Выбран таргет: target_10_1  (positive rate = 31.505%)


In [6]:
def prepare_sample(X_df, target_series, n=SAMPLE_N, random_state=RANDOM_STATE):
    """Объединяем фичи с таргетом, берём сэмпл, делаем train/val split."""
    df = X_df.join(target_series, how='inner')
    df = df.sample(min(n, len(df)), random_state=random_state)
    y = df[TARGET].values
    X = df.drop(columns=[TARGET])
    return train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)

y_series = df_target.set_index(ID_COL)[TARGET]

X_tr_nan,  X_val_nan,  y_tr, y_val = prepare_sample(tree_nan_train.set_index(ID_COL),  y_series)
X_tr_zero, X_val_zero, _,    _     = prepare_sample(tree_zero_train.set_index(ID_COL), y_series)
X_tr_lin,  X_val_lin,  _,    _     = prepare_sample(linear_train.set_index(ID_COL),    y_series)

print(f'Train size: {len(y_tr)}  |  Val size: {len(y_val)}')
print(f'Positive rate (train): {y_tr.mean():.3%}')

Train size: 40000  |  Val size: 10000
Positive rate (train): 31.655%


###  LightGBM (tree_nan)

In [9]:
lgb_params = dict(
    objective='binary',
    metric='auc',
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbose=-1,
    random_state=RANDOM_STATE,
)

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_tr_nan, y_tr,
    eval_set=[(X_val_nan, y_val)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(50)]
)

lgb_auc = roc_auc_score(y_val, lgb_model.predict_proba(X_val_nan)[:, 1])
print(f'LightGBM  ROC-AUC = {lgb_auc:.4f}  (best iter: {lgb_model.best_iteration_})')

[50]	valid_0's auc: 0.742155
[100]	valid_0's auc: 0.743906
LightGBM  ROC-AUC = 0.7445  (best iter: 110)


### CatBoost (tree_nan)

In [10]:
# CatBoost принимает категориальные признаки напрямую
cat_feature_names = [c for c in X_tr_nan.columns if c.startswith('cat_feature_')]
cat_feature_idx   = [X_tr_nan.columns.get_loc(c) for c in cat_feature_names]

# CatBoost требует, чтобы кат. признаки были строками или int (не float)
X_tr_nan_cb  = X_tr_nan.copy()
X_val_nan_cb = X_val_nan.copy()
for c in cat_feature_names:
    X_tr_nan_cb[c]  = X_tr_nan_cb[c].astype(str)
    X_val_nan_cb[c] = X_val_nan_cb[c].astype(str)

cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    cat_features=cat_feature_idx,
    early_stopping_rounds=30,
    random_seed=RANDOM_STATE,
    verbose=50,
    task_type="GPU",
)
cb_model.fit(
    X_tr_nan_cb, y_tr,
    eval_set=(X_val_nan_cb, y_val),
)

cb_auc = roc_auc_score(y_val, cb_model.predict_proba(X_val_nan_cb)[:, 1])
print(f'CatBoost  ROC-AUC = {cb_auc:.4f}')

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7004189	best: 0.7004189 (0)	total: 226ms	remaining: 1m 7s
50:	test: 0.7353950	best: 0.7353950 (50)	total: 4.82s	remaining: 23.5s
100:	test: 0.7389815	best: 0.7390005 (98)	total: 9.28s	remaining: 18.3s
150:	test: 0.7404383	best: 0.7404383 (150)	total: 13.6s	remaining: 13.4s
200:	test: 0.7409025	best: 0.7410130 (194)	total: 17.7s	remaining: 8.71s
250:	test: 0.7411925	best: 0.7412030 (249)	total: 21.9s	remaining: 4.27s
299:	test: 0.7419600	best: 0.7420283 (297)	total: 25.9s	remaining: 0us
bestTest = 0.742028296
bestIteration = 297
Shrink model to first 298 iterations.
CatBoost  ROC-AUC = 0.7420


###  XGBoost (tree_zero)

In [11]:
# tree_zero: NaN заменены на 0, добавлены _isnan флаги
# Кат. признаки уже закодированы как числа (ordinal) из оригинального датасета
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    verbosity=0,
)
xgb_model.fit(
    X_tr_zero, y_tr,
    eval_set=[(X_val_zero, y_val)],
    verbose=50,
)

xgb_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val_zero)[:, 1])
print(f'XGBoost   ROC-AUC = {xgb_auc:.4f}  (best iter: {xgb_model.best_iteration})')

[0]	validation_0-auc:0.71296
[50]	validation_0-auc:0.74034
[100]	validation_0-auc:0.74399
[150]	validation_0-auc:0.74456
[200]	validation_0-auc:0.74501
[231]	validation_0-auc:0.74457
XGBoost   ROC-AUC = 0.7452  (best iter: 201)


##  Logreg

In [7]:
# linear: QT + OHE + isnan флаги, нет NaN
logreg = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
logreg.fit(X_tr_lin, y_tr)

lr_auc = roc_auc_score(y_val, logreg.predict_proba(X_val_lin)[:, 1])
print(f'LogReg    ROC-AUC = {lr_auc:.4f}')

LogReg    ROC-AUC = 0.6707


# Все отработало

# Подбор модели и параметров

In [4]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X_full = tree_nan_train.set_index(ID_COL)
y_full = df_target.set_index(ID_COL)[target_cols]

common_idx = X_full.index.intersection(y_full.index)
X_full = X_full.loc[common_idx]
y_full = y_full.loc[common_idx]

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_pos, val_pos = next(msss.split(X_full, y_full))

train_idx = common_idx[train_pos]
val_idx   = common_idx[val_pos]

X_tr      = X_full.loc[train_idx]
X_val_opt = X_full.loc[val_idx]

print(f'Train: {len(train_idx):,}  Val: {len(val_idx):,}')
print('Проверка баланса (positive rate train vs val):')
check = pd.DataFrame({
    'train': y_full.loc[train_idx].mean(),
    'val':   y_full.loc[val_idx].mean(),
})
check['diff'] = (check['train'] - check['val']).abs()
print(check.sort_values('diff', ascending=False).head(10).to_string())

Train: 599,926  Val: 150,074
Проверка баланса (positive rate train vs val):
                train       val      diff
target_10_1  0.315091  0.314898  0.000193
target_9_6   0.223099  0.222963  0.000136
target_8_1   0.102509  0.102443  0.000067
target_3_2   0.097422  0.097359  0.000063
target_3_1   0.098385  0.098325  0.000061
target_9_7   0.077258  0.077209  0.000049
target_7_1   0.062508  0.062469  0.000039
target_9_2   0.036454  0.036429  0.000026
target_8_2   0.032534  0.032511  0.000023
target_1_4   0.023433  0.023415  0.000018


## подбор гиперпараметров LightGBM по всем таргетам



In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 50
N_JOBS   = 8  

def run_study(tgt):
    y_tr_arr  = y_full.loc[train_idx, tgt].values
    y_val_arr = y_full.loc[val_idx,   tgt].values

    if y_tr_arr.sum() == 0 or y_val_arr.sum() == 0:
        return tgt, None

    def objective(trial):
        params = dict(
            objective         = 'binary',
            metric            = 'auc',
            verbose           = -1,
            num_threads       = 2,
            n_estimators      = trial.suggest_int  ('n_estimators',      100, 2000, log=True),
            learning_rate     = trial.suggest_float('learning_rate',    1e-3,  0.3, log=True),
            num_leaves        = trial.suggest_int  ('num_leaves',         16,  512, log=True),
            max_depth         = trial.suggest_int  ('max_depth',           3,   12),
            min_child_samples = trial.suggest_int  ('min_child_samples',   5,  200, log=True),
            feature_fraction  = trial.suggest_float('feature_fraction',  0.4,  1.0),
            lambda_l1         = trial.suggest_float('lambda_l1',        1e-8, 10.0, log=True),
            lambda_l2         = trial.suggest_float('lambda_l2',        1e-8, 10.0, log=True),
            random_state      = RANDOM_STATE,
        )
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr_arr)
        return roc_auc_score(y_val_arr, model.predict_proba(X_val_opt)[:, 1])

    study = optuna.create_study(
        direction = 'maximize',
        sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner    = optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    return tgt, {'best_auc': study.best_value, 'best_params': study.best_params}

raw = Parallel(n_jobs=N_JOBS, backend='loky')(
    delayed(run_study)(tgt) for tgt in tqdm(target_cols, desc='Targets')
)

all_results = {tgt: res for tgt, res in raw if res is not None}
print(f'Готово. Обработано таргетов: {len(all_results)}')

Targets:   0%|          | 0/41 [00:00<?, ?it/s]

Готово. Обработано таргетов: 41


In [ ]:
# Сохраняем результаты
with open('lgb_best_params.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('Сохранено: lgb_best_params.json')

# Сводная таблица
summary = pd.DataFrame({
    tgt: {'best_auc': v['best_auc'], **v['best_params']}
    for tgt, v in all_results.items()
}).T.sort_values('best_auc', ascending=False)

summary['best_auc'] = summary['best_auc'].astype(float).map('{:.4f}'.format)
print(f'ROC-AUC по таргетам')
print(summary[['best_auc']].to_string())
print(f'Средний AUC: {summary["best_auc"].astype(float).mean():.4f}')

Сохранено: lgb_best_params.json
ROC-AUC по таргетам
            best_auc
target_8_1    0.9761
target_3_5    0.9548
target_2_2    0.9169
target_9_8    0.9141
target_3_2    0.9019
target_3_4    0.8983
target_2_8    0.8796
target_1_1    0.8772
target_6_5    0.8750
target_9_4    0.8615
target_1_5    0.8449
target_8_3    0.8421
target_1_3    0.8373
target_7_2    0.8213
target_8_2    0.8184
target_4_1    0.8054
target_9_2    0.7950
target_1_4    0.7897
target_7_1    0.7835
target_6_4    0.7756
target_1_2    0.7583
target_9_5    0.7539
target_2_1    0.7458
target_10_1   0.7456
target_2_3    0.7296
target_9_7    0.7295
target_9_1    0.7125
target_7_3    0.6969
target_2_5    0.6944
target_6_3    0.6848
target_2_7    0.6814
target_3_3    0.6750
target_9_6    0.6720
target_3_1    0.6701
target_2_4    0.6685
target_5_1    0.6577
target_5_2    0.6564
target_2_6    0.6522
target_6_2    0.6236
target_9_3    0.6211
target_6_1    0.6063
Средний AUC: 0.7708


## подбор гиперпараметров CatBoost по всем таргетам

In [ ]:
# tree_nan признаки переводим в строки
cat_feature_names = [c for c in X_full.columns if c.startswith('cat_feature_')]
cat_feature_idx   = [X_full.columns.get_loc(c) for c in cat_feature_names]

X_tr_cb  = X_tr.copy()
X_val_cb = X_val_opt.copy()
for c in cat_feature_names:
    X_tr_cb[c]  = X_tr_cb[c].astype(str)
    X_val_cb[c] = X_val_cb[c].astype(str)

print(f'CatBoost train: {X_tr_cb.shape}  val: {X_val_cb.shape}')
print(f'cat_features: {len(cat_feature_names)}')

CatBoost train: (599926, 207)  val: (150074, 207)
cat_features: 67


In [ ]:
N_TRIALS_CB = 1
N_JOBS_CB   = 2   

def run_study_cb(tgt):
    y_tr_arr  = y_full.loc[train_idx, tgt].values
    y_val_arr = y_full.loc[val_idx,   tgt].values

    if y_tr_arr.sum() == 0 or y_val_arr.sum() == 0:
        return tgt, None

    def objective(trial):
        params = dict(
            iterations      = trial.suggest_int  ('iterations',      100, 2000, log=True),
            learning_rate   = trial.suggest_float('learning_rate',  1e-3,  0.3, log=True),
            depth           = trial.suggest_int  ('depth',             3,   10),
            l2_leaf_reg     = trial.suggest_float('l2_leaf_reg',     1e-8, 10.0, log=True),
            random_strength = trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
            border_count    = trial.suggest_int  ('border_count',      32,  255),
            eval_metric     = 'AUC',
            loss_function   = 'Logloss',
            cat_features    = cat_feature_idx,
            task_type       = 'GPU',
            thread_count    = 2,
            random_seed     = RANDOM_STATE,
            verbose         = False,
        )
        model = CatBoostClassifier(**params)
        model.fit(X_tr_cb, y_tr_arr)
        return roc_auc_score(y_val_arr, model.predict_proba(X_val_cb)[:, 1])

    study = optuna.create_study(
        direction = 'maximize',
        sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=N_TRIALS_CB, show_progress_bar=False)
    return tgt, {'best_auc': study.best_value, 'best_params': study.best_params}

raw_cb = Parallel(n_jobs=N_JOBS_CB, backend='loky')(
    delayed(run_study_cb)(tgt) for tgt in tqdm(target_cols, desc='Targets CB')
)

cb_results = {tgt: res for tgt, res in raw_cb if res is not None}
print(f'\nГотово. Обработано таргетов: {len(cb_results)}')

Targets CB:   0%|          | 0/41 [00:00<?, ?it/s]

Error: Error: KeyboardInterrupt: 
[31m---------------------------------------------------------------------------[39m
[31mKeyboardInterrupt[39m                         Traceback (most recent call last)
[36mCell[39m[36m [39m[32mIn[9][39m[32m, line 39[39m
[32m     36[39m     study.optimize(objective, n_trials=N_TRIALS_CB, show_progress_bar=[38;5;28;01mFalse[39;00m)
[32m     37[39m     [38;5;28;01mreturn[39;00m tgt, {[33m'[39m[33mbest_auc[39m[33m'[39m: study.best_value, [33m'[39m[33mbest_params[39m[33m'[39m: study.best_params}
[32m---> [39m[32m39[39m raw_cb = [43mParallel[49m[43m([49m[43mn_jobs[49m[43m=[49m[43mN_JOBS_CB[49m[43m,[49m[43m [49m[43mbackend[49m[43m=[49m[33;43m'[39;49m[33;43mloky[39;49m[33;43m'[39;49m[43m)[49m[43m([49m
[32m     40[39m [43m    [49m[43mdelayed[49m[43m([49m[43mrun_study_cb[49m[43m)[49m[43m([49m[43mtgt[49m[43m)[49m[43m [49m[38;5;28;43;01mfor[39;49;00m[43m [49m[43mtgt[49m[43m 

### не хватит мощностей, чтобы успеть

In [ ]:
with open('cb_best_params.json', 'w') as f:
    json.dump(cb_results, f, indent=2)
print('Сохранено: cb_best_params.json')

summary_cb = pd.DataFrame({
    tgt: {'best_auc': v['best_auc'], **v['best_params']}
    for tgt, v in cb_results.items()
}).T.sort_values('best_auc', ascending=False)

summary_cb['best_auc'] = summary_cb['best_auc'].astype(float).map('{:.4f}'.format)
print(f'ROC-AUC по таргетам')
print(summary_cb[['best_auc']].to_string())
print(f'Средний AUC: {summary_cb["best_auc"].astype(float).mean():.4f}')

### CatBoost один набор параметров

In [ ]:
cb_fixed_params = dict(
    iterations    = 1000,
    learning_rate = 0.05,
    depth         = 6,
    l2_leaf_reg   = 3.0,
    #eval_metric   = 'AUC',
    loss_function = 'Logloss',
    cat_features  = cat_feature_idx,
    task_type     = 'GPU',
    random_seed   = RANDOM_STATE,
    verbose       = True,
)

cb_fixed_results = {}
for tgt in tqdm(target_cols, desc='CatBoost'):
    y_tr_arr  = y_full.loc[train_idx, tgt].values
    y_val_arr = y_full.loc[val_idx,   tgt].values


    model = CatBoostClassifier(**cb_fixed_params)
    model.fit(X_tr_cb, y_tr_arr)
    auc = roc_auc_score(y_val_arr, model.predict_proba(X_val_cb)[:, 1])
    cb_fixed_results[tgt] = {'best_auc': auc, 'best_params': cb_fixed_params}

with open('cb_fixed_params.json', 'w') as f:
    json.dump(cb_fixed_results, f, indent=2)
print('Сохранено: cb_fixed_params.json')

summary_cb_fixed = pd.Series({t: v['best_auc'] for t, v in cb_fixed_results.items()}).sort_values(ascending=False)
print(summary_cb_fixed.map('{:.4f}'.format).to_string())
print(f'\nСредний AUC: {summary_cb_fixed.mean():.4f}')

CatBoost:   0%|          | 0/41 [00:00<?, ?it/s]

0:	learn: 0.5811509	total: 383ms	remaining: 6m 22s
1:	learn: 0.4864996	total: 551ms	remaining: 4m 34s
2:	learn: 0.4065187	total: 692ms	remaining: 3m 49s
3:	learn: 0.3464302	total: 855ms	remaining: 3m 32s
4:	learn: 0.2944220	total: 985ms	remaining: 3m 16s
5:	learn: 0.2548878	total: 1.11s	remaining: 3m 3s
6:	learn: 0.2216761	total: 1.24s	remaining: 2m 55s
7:	learn: 0.1922605	total: 1.37s	remaining: 2m 49s
8:	learn: 0.1694914	total: 1.51s	remaining: 2m 46s
9:	learn: 0.1503055	total: 1.66s	remaining: 2m 44s
10:	learn: 0.1340610	total: 1.78s	remaining: 2m 40s
11:	learn: 0.1206764	total: 1.91s	remaining: 2m 37s
12:	learn: 0.1089600	total: 2.05s	remaining: 2m 35s
13:	learn: 0.0995765	total: 2.19s	remaining: 2m 34s
14:	learn: 0.0916550	total: 2.3s	remaining: 2m 31s
15:	learn: 0.0847765	total: 2.44s	remaining: 2m 29s
16:	learn: 0.0798548	total: 2.58s	remaining: 2m 28s
17:	learn: 0.0756757	total: 2.7s	remaining: 2m 27s
18:	learn: 0.0713951	total: 2.83s	remaining: 2m 26s
19:	learn: 0.0678914	tota

CatBoost:   2%|▏         | 1/41 [02:15<1:30:08, 135.20s/it]

0:	learn: 0.5580719	total: 124ms	remaining: 2m 3s
1:	learn: 0.4486983	total: 242ms	remaining: 2m
2:	learn: 0.3628459	total: 352ms	remaining: 1m 56s
3:	learn: 0.2935980	total: 467ms	remaining: 1m 56s
4:	learn: 0.2387000	total: 575ms	remaining: 1m 54s
5:	learn: 0.1956018	total: 709ms	remaining: 1m 57s
6:	learn: 0.1618417	total: 817ms	remaining: 1m 55s
7:	learn: 0.1350986	total: 927ms	remaining: 1m 54s
8:	learn: 0.1143198	total: 1.04s	remaining: 1m 55s
9:	learn: 0.0976919	total: 1.16s	remaining: 1m 54s
10:	learn: 0.0844436	total: 1.27s	remaining: 1m 54s
11:	learn: 0.0738838	total: 1.39s	remaining: 1m 54s
12:	learn: 0.0652684	total: 1.52s	remaining: 1m 55s
13:	learn: 0.0582867	total: 1.62s	remaining: 1m 54s
14:	learn: 0.0525436	total: 1.74s	remaining: 1m 54s
15:	learn: 0.0478612	total: 1.84s	remaining: 1m 53s
16:	learn: 0.0439278	total: 1.96s	remaining: 1m 53s
17:	learn: 0.0406952	total: 2.07s	remaining: 1m 52s
18:	learn: 0.0379286	total: 2.18s	remaining: 1m 52s
19:	learn: 0.0356739	total:

CatBoost:   5%|▍         | 2/41 [04:21<1:24:37, 130.20s/it]

...

0:	learn: 0.5668106	total: 187ms	remaining: 3m 6s
1:	learn: 0.4740587	total: 365ms	remaining: 3m 2s
2:	learn: 0.4015906	total: 549ms	remaining: 3m 2s
3:	learn: 0.3377424	total: 742ms	remaining: 3m 4s
4:	learn: 0.2878614	total: 917ms	remaining: 3m 2s
5:	learn: 0.2471657	total: 1.09s	remaining: 3m 1s
6:	learn: 0.2133577	total: 1.29s	remaining: 3m 3s
7:	learn: 0.1858473	total: 1.48s	remaining: 3m 3s
8:	learn: 0.1615299	total: 1.67s	remaining: 3m 3s
9:	learn: 0.1432166	total: 1.84s	remaining: 3m 1s
10:	learn: 0.1288392	total: 2s	remaining: 2m 59s
11:	learn: 0.1164762	total: 2.2s	remaining: 3m 1s
12:	learn: 0.1066188	total: 2.38s	remaining: 3m
13:	learn: 0.0986107	total: 2.56s	remaining: 3m
14:	learn: 0.0905445	total: 2.75s	remaining: 3m
15:	learn: 0.0848991	total: 2.92s	remaining: 2m 59s
16:	learn: 0.0778245	total: 3.1s	remaining: 2m 59s
17:	learn: 0.0733183	total: 3.28s	remaining: 2m 58s
18:	learn: 0.0696307	total: 3.46s	remaining: 2m 58s
19:	learn: 0.0667372	total: 3.64s	remaining: 2m 58

CatBoost:  98%|█████████▊| 40/41 [2:10:18<03:21, 201.90s/it]

0:	learn: 0.6779880	total: 322ms	remaining: 5m 21s
1:	learn: 0.6643675	total: 482ms	remaining: 4m
2:	learn: 0.6523973	total: 639ms	remaining: 3m 32s
3:	learn: 0.6417884	total: 802ms	remaining: 3m 19s
4:	learn: 0.6331408	total: 966ms	remaining: 3m 12s
5:	learn: 0.6254532	total: 1.11s	remaining: 3m 4s
6:	learn: 0.6182887	total: 1.28s	remaining: 3m 1s
7:	learn: 0.6114080	total: 1.45s	remaining: 2m 59s
8:	learn: 0.6050370	total: 1.63s	remaining: 2m 59s
9:	learn: 0.5993735	total: 1.8s	remaining: 2m 58s
10:	learn: 0.5944741	total: 1.97s	remaining: 2m 56s
11:	learn: 0.5901779	total: 2.15s	remaining: 2m 56s
12:	learn: 0.5862773	total: 2.32s	remaining: 2m 56s
13:	learn: 0.5829289	total: 2.5s	remaining: 2m 55s
14:	learn: 0.5799031	total: 2.67s	remaining: 2m 55s
15:	learn: 0.5768039	total: 2.83s	remaining: 2m 54s
16:	learn: 0.5739878	total: 3s	remaining: 2m 53s
17:	learn: 0.5716727	total: 3.17s	remaining: 2m 53s
18:	learn: 0.5695491	total: 3.35s	remaining: 2m 53s
19:	learn: 0.5673531	total: 3.52s

CatBoost: 100%|██████████| 41/41 [2:13:32<00:00, 195.42s/it]


Error: Error: NameError: name 'json' is not defined
[31m---------------------------------------------------------------------------[39m
[31mNameError[39m                                 Traceback (most recent call last)
[36mCell[39m[36m [39m[32mIn[8][39m[32m, line 28[39m
[32m     25[39m     cb_fixed_results[tgt] = {[33m'[39m[33mbest_auc[39m[33m'[39m: auc, [33m'[39m[33mbest_params[39m[33m'[39m: cb_fixed_params}
[32m     27[39m [38;5;28;01mwith[39;00m [38;5;28mopen[39m([33m'[39m[33mcb_fixed_params.json[39m[33m'[39m, [33m'[39m[33mw[39m[33m'[39m) [38;5;28;01mas[39;00m f:
[32m---> [39m[32m28[39m     [43mjson[49m.dump(cb_fixed_results, f, indent=[32m2[39m)
[32m     29[39m [38;5;28mprint[39m([33m'[39m[33mСохранено: cb_fixed_params.json[39m[33m'[39m)
[32m     31[39m summary_cb_fixed = pd.Series({t: v[[33m'[39m[33mbest_auc[39m[33m'[39m] [38;5;28;01mfor[39;00m t, v [38;5;129;01min[39;00m cb_fixed_results.items()}).sort_v

In [ ]:
import json
with open('cb_fixed_params.json', 'w') as f:
    json.dump(cb_fixed_results, f, indent=2)
print('Сохранено: cb_fixed_params.json')

summary_cb_fixed = pd.Series({t: v['best_auc'] for t, v in cb_fixed_results.items()}).sort_values(ascending=False)
print(summary_cb_fixed.map('{:.4f}'.format).to_string())
print(f'\nСредний AUC: {summary_cb_fixed.mean():.4f}')

Сохранено: cb_fixed_params.json
target_8_1     0.9764
target_2_8     0.9728
target_3_5     0.9715
target_6_5     0.9472
target_3_4     0.9304
target_2_2     0.9277
target_9_8     0.9240
target_3_2     0.9064
target_1_1     0.9054
target_9_4     0.8990
target_8_3     0.8674
target_1_5     0.8661
target_1_3     0.8552
target_2_7     0.8522
target_6_4     0.8502
target_4_1     0.8477
target_7_2     0.8379
target_8_2     0.8269
target_9_5     0.8251
target_9_2     0.8215
target_1_4     0.8192
target_2_1     0.8069
target_1_2     0.8028
target_9_1     0.7998
target_7_1     0.7980
target_2_3     0.7927
target_7_3     0.7757
target_3_3     0.7672
target_2_5     0.7586
target_6_3     0.7564
target_5_2     0.7521
target_10_1    0.7510
target_9_7     0.7480
target_5_1     0.7473
target_2_6     0.7419
target_2_4     0.7349
target_6_2     0.7296
target_6_1     0.7145
target_3_1     0.6870
target_9_6     0.6838
target_9_3     0.6814

Средний AUC: 0.8210


## подбор гиперпараметров XGBoost по всем таргетам

In [10]:
# tree_zero 
X_full_zero = tree_zero_train.set_index(ID_COL).loc[common_idx]

X_tr_xgb  = X_full_zero.loc[train_idx]
X_val_xgb = X_full_zero.loc[val_idx]

print(f'XGBoost train: {X_tr_xgb.shape}  val: {X_val_xgb.shape}')

XGBoost train: (599926, 339)  val: (150074, 339)


In [13]:
N_TRIALS_XGB = 1
N_JOBS_XGB   = 8

def run_study_xgb(tgt):
    y_tr_arr  = y_full.loc[train_idx, tgt].values
    y_val_arr = y_full.loc[val_idx,   tgt].values



    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int  ('n_estimators',      100, 2000, log=True),
            learning_rate     = trial.suggest_float('learning_rate',    1e-3,  0.3, log=True),
            max_depth         = trial.suggest_int  ('max_depth',           3,   12),
            min_child_weight  = trial.suggest_float('min_child_weight',    1, 100.0, log=True),
            subsample         = trial.suggest_float('subsample',          0.4,  1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree',   0.4,  1.0),
            reg_lambda        = trial.suggest_float('reg_lambda',        1e-8, 10.0, log=True),
            reg_alpha         = trial.suggest_float('reg_alpha',         1e-8, 10.0, log=True),
            eval_metric       = 'auc',
            objective         = 'binary:logistic',
            nthread           = 2,
            verbosity         = 0,
            random_state      = RANDOM_STATE,
        )
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr_xgb, y_tr_arr)
        return roc_auc_score(y_val_arr, model.predict_proba(X_val_xgb)[:, 1])

    study = optuna.create_study(
        direction = 'maximize',
        sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=N_TRIALS_XGB, show_progress_bar=False)
    return tgt, {'best_auc': study.best_value, 'best_params': study.best_params}

raw_xgb = Parallel(n_jobs=N_JOBS_XGB, backend='loky')(
    delayed(run_study_xgb)(tgt) for tgt in tqdm(target_cols, desc='Targets XGB')
)

xgb_results = {tgt: res for tgt, res in raw_xgb if res is not None}
print(f'Готово. Обработано таргетов: {len(xgb_results)}')

Targets XGB:   0%|          | 0/41 [00:00<?, ?it/s]


Готово. Обработано таргетов: 41


In [14]:
with open('xgb_best_params.json', 'w') as f:
    json.dump(xgb_results, f, indent=2)
print('Сохранено: xgb_best_params.json')

summary_xgb = pd.DataFrame({
    tgt: {'best_auc': v['best_auc'], **v['best_params']}
    for tgt, v in xgb_results.items()
}).T.sort_values('best_auc', ascending=False)

summary_xgb['best_auc'] = summary_xgb['best_auc'].astype(float).map('{:.4f}'.format)
print('ROC-AUC по таргетам')
print(summary_xgb[['best_auc']].to_string())
print(f'Средний AUC: {summary_xgb["best_auc"].astype(float).mean():.4f}')

Сохранено: xgb_best_params.json
ROC-AUC по таргетам
            best_auc
target_8_1    0.9735
target_3_5    0.9607
target_2_2    0.9133
target_6_5    0.9117
target_9_8    0.9037
target_3_4    0.8992
target_3_2    0.8953
target_2_8    0.8831
target_1_1    0.8785
target_9_4    0.8561
target_8_3    0.8444
target_1_5    0.8352
target_1_3    0.8296
target_7_2    0.8121
target_8_2    0.8034
target_2_7    0.8018
target_4_1    0.8001
target_6_4    0.7924
target_9_2    0.7819
target_1_4    0.7798
target_7_1    0.7656
target_1_2    0.7545
target_9_5    0.7427
target_2_3    0.7408
target_2_1    0.7363
target_10_1   0.7358
target_7_3    0.7143
target_9_7    0.7134
target_2_5    0.7127
target_9_1    0.7116
target_3_3    0.7030
target_6_3    0.6815
target_2_6    0.6789
target_2_4    0.6759
target_5_1    0.6680
target_9_6    0.6564
target_6_2    0.6505
target_5_2    0.6487
target_3_1    0.6471
target_6_1    0.6442
target_9_3    0.6189
Средний AUC: 0.7746


## Стандартные параметры 

In [17]:
lgb_stand_params = dict(
    objective         = 'binary',
    metric            = 'auc',
    verbose           = -1,
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 64,
    max_depth         = -1,
    min_child_samples = 20,
    feature_fraction  = 0.8,
    lambda_l1         = 0.1,
    lambda_l2         = 0.1,
    random_state      = RANDOM_STATE,
)

lgb_stand_results = {}
for tgt in tqdm(target_cols, desc='LGB standard'):
    y_tr_arr  = y_full.loc[train_idx, tgt].values
    y_val_arr = y_full.loc[val_idx,   tgt].values

    if y_tr_arr.sum() == 0 or y_val_arr.sum() == 0:
        continue

    model = lgb.LGBMClassifier(**lgb_stand_params)
    model.fit(X_tr, y_tr_arr)
    auc = roc_auc_score(y_val_arr, model.predict_proba(X_val_opt)[:, 1])
    lgb_stand_results[tgt] = {'best_auc': auc, 'best_params': lgb_stand_params}

with open('lgb_stand_params.json', 'w') as f:
    json.dump(lgb_stand_results, f, indent=2)
print('Сохранено: lgb_stand_params.json')

summary_lgb_stand = pd.Series({t: v['best_auc'] for t, v in lgb_stand_results.items()}).sort_values(ascending=False)
print(summary_lgb_stand.map('{:.4f}'.format).to_string())
print(f'Средний AUC: {summary_lgb_stand.mean():.4f}')

LGB standard:   0%|          | 0/41 [00:00<?, ?it/s]

Сохранено: lgb_stand_params.json
target_8_1     0.9780
target_3_5     0.9680
target_2_8     0.9316
target_2_2     0.9314
target_9_8     0.9261
target_6_5     0.9216
target_3_4     0.9113
target_3_2     0.9074
target_1_1     0.9070
target_9_4     0.8860
target_1_5     0.8721
target_8_3     0.8702
target_1_3     0.8590
target_7_2     0.8463
target_4_1     0.8393
target_6_4     0.8373
target_8_2     0.8341
target_1_4     0.8228
target_9_2     0.8199
target_9_5     0.8106
target_7_1     0.7988
target_2_1     0.7920
target_2_7     0.7899
target_1_2     0.7845
target_2_3     0.7793
target_9_1     0.7687
target_10_1    0.7529
target_2_5     0.7512
target_9_7     0.7465
target_7_3     0.7409
target_6_3     0.7348
target_5_1     0.7333
target_2_4     0.7198
target_2_6     0.7043
target_5_2     0.7037
target_6_2     0.7032
target_3_3     0.7010
target_3_1     0.6867
target_6_1     0.6862
target_9_6     0.6840
target_9_3     0.6653
Средний AUC: 0.8075


In [18]:
xgb_stand_params = dict(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = RANDOM_STATE,
    verbosity         = 0,
)

X_tr_zero_full   = tree_zero_train.set_index(ID_COL).loc[train_idx]
X_val_zero_full  = tree_zero_train.set_index(ID_COL).loc[val_idx]

xgb_stand_results = {}
for tgt in tqdm(target_cols, desc='XGB standard'):
    y_tr_arr  = y_full.loc[train_idx, tgt].values
    y_val_arr = y_full.loc[val_idx,   tgt].values

    if y_tr_arr.sum() == 0 or y_val_arr.sum() == 0:
        continue

    model = xgb.XGBClassifier(**xgb_stand_params)
    model.fit(X_tr_zero_full, y_tr_arr)
    auc = roc_auc_score(y_val_arr, model.predict_proba(X_val_zero_full)[:, 1])
    xgb_stand_results[tgt] = {'best_auc': auc, 'best_params': xgb_stand_params}

with open('xgb_stand_params.json', 'w') as f:
    json.dump(xgb_stand_results, f, indent=2)
print('Сохранено: xgb_stand_params.json')

summary_xgb_stand = pd.Series({t: v['best_auc'] for t, v in xgb_stand_results.items()}).sort_values(ascending=False)
print(summary_xgb_stand.map('{:.4f}'.format).to_string())
print(f'Средний AUC: {summary_xgb_stand.mean():.4f}')

XGB standard:   0%|          | 0/41 [00:00<?, ?it/s]

Сохранено: xgb_stand_params.json
target_8_1     0.9778
target_3_5     0.9694
target_2_8     0.9423
target_2_2     0.9330
target_9_8     0.9291
target_6_5     0.9194
target_3_4     0.9148
target_1_1     0.9094
target_3_2     0.9076
target_9_4     0.8888
target_8_3     0.8735
target_1_5     0.8724
target_1_3     0.8625
target_7_2     0.8484
target_4_1     0.8472
target_6_4     0.8414
target_8_2     0.8348
target_1_4     0.8260
target_9_2     0.8217
target_9_5     0.8145
target_7_1     0.8023
target_2_1     0.7995
target_1_2     0.7990
target_2_7     0.7987
target_9_1     0.7783
target_2_3     0.7731
target_7_3     0.7682
target_2_5     0.7557
target_10_1    0.7518
target_9_7     0.7480
target_6_3     0.7436
target_5_1     0.7406
target_2_4     0.7301
target_2_6     0.7253
target_6_2     0.7162
target_5_2     0.7129
target_6_1     0.6988
target_3_3     0.6921
target_3_1     0.6877
target_9_6     0.6843
target_9_3     0.6766
Средний AUC: 0.8126


## Финальное обучение и сабмит

Для каждого таргета выбирается лучшая модель  из трёх JSON-файлов.  

In [19]:
with open('lgb_stand_params.json')   as f: lgb_results = json.load(f)
with open('cb_fixed_params.json')   as f: cb_results  = json.load(f)
with open('xgb_stand_params.json')   as f: xgb_results = json.load(f)

model_registry = {
    'lgb': lgb_results,
    'cb':  cb_results,
    'xgb': xgb_results,
}

best_model_per_target = {}
for tgt in target_cols:
    candidates = {name: res[tgt]['best_auc']
                  for name, res in model_registry.items()
                  if tgt in res}
    if not candidates:
        continue
    best_name = max(candidates, key=candidates.get)
    best_model_per_target[tgt] = {
        'model':      best_name,
        'best_auc':   candidates[best_name],
        'best_params': model_registry[best_name][tgt]['best_params'],
    }

summary_best = pd.DataFrame(best_model_per_target).T[['model', 'best_auc']].sort_values('best_auc', ascending=False)
summary_best['best_auc'] = summary_best['best_auc'].astype(float).map('{:.4f}'.format)
summary_best

,model,best_auc
target_8_1,lgb,0.9780
target_2_8,cb,0.9728
target_3_5,cb,0.9715
target_6_5,cb,0.9472
target_2_2,xgb,0.9330
target_3_4,cb,0.9304
target_9_8,xgb,0.9291
target_1_1,xgb,0.9094
target_3_2,xgb,0.9076
target_9_4,cb,0.8990


In [ ]:
tree_nan_test  = pd.read_parquet('tree_nan_test.parquet').set_index(ID_COL)
tree_zero_test = pd.read_parquet('tree_zero_test.parquet').set_index(ID_COL)

cat_feature_names = [c for c in X_full.columns if c.startswith('cat_feature_')]
cat_feature_idx   = [X_full.columns.get_loc(c) for c in cat_feature_names]

def to_str_cats(df):
    df = df.copy()
    for c in cat_feature_names:
        df[c] = df[c].astype(str)
    return df

X_full_cb      = to_str_cats(X_full)
tree_nan_test_cb = to_str_cats(tree_nan_test)


y_full_all = df_target.set_index(ID_COL)[target_cols].loc[common_idx]

print(f'Train full: {X_full.shape}')
print(f'Test  nan : {tree_nan_test.shape}')
print(f'Test  zero: {tree_zero_test.shape}')

In [ ]:
submission = pd.DataFrame(index=tree_nan_test.index)

for tgt in tqdm(target_cols, desc='Final train'):
    if tgt not in best_model_per_target:
        submission[tgt] = 0.0
        continue

    info   = best_model_per_target[tgt]
    name   = info['model']
    params = info['best_params']
    y_tr   = y_full_all[tgt].values

    if name == 'lgb':
        model = lgb.LGBMClassifier(**params, verbose=-1)
        model.fit(X_full, y_tr)
        preds = model.predict_proba(tree_nan_test)[:, 1]

    elif name == 'cb':
        cb_params = {k: v for k, v in params.items() if k != 'cat_features'}
        model = CatBoostClassifier(**cb_params, cat_features=cat_feature_idx, verbose=False)
        model.fit(X_full_cb, y_tr)
        preds = model.predict_proba(tree_nan_test_cb)[:, 1]

    elif name == 'xgb':
        model = xgb.XGBClassifier(**params, verbosity=0)
        model.fit(tree_zero_test.reindex(X_full.index), y_tr)   
        preds = model.predict_proba(tree_zero_test)[:, 1]

    submission[tgt] = preds

submission.index.name = ID_COL
submission = submission.reset_index()
submission.to_csv('submission.csv', index=False)
print(f'Сохранено: submission.csv  {submission.shape}')
submission.head()